# 🏛️ LegalVision: Fine-tuning Llama 3 for Legal Reasoning
## Using Unsloth QLoRA - 2x Faster, 60% Less Memory

**Author:** S. Sivanuja  
**Project:** LegalVision - Explainable Legal Reasoning Module  
**Model:** Llama 3.1 8B (4-bit Quantized)  

---

### 📋 What This Notebook Does:
1. Installs Unsloth and dependencies
2. Loads your legal reasoning dataset
3. Fine-tunes Llama 3.1 8B with QLoRA
4. Saves the model (LoRA adapters + full merge options)
5. Tests the fine-tuned model

### ⚙️ Requirements:
- Google Colab with GPU (T4 free tier works!)
- Your processed dataset (`finetuning_data.jsonl` or `reasoning_entries.json`)
- ~12GB VRAM (T4 has 15GB)

---

## 📦 Step 1: Install Unsloth and Dependencies

This cell installs Unsloth with all optimizations. Takes ~2-3 minutes.

In [ ]:
%%capture
# Install Unsloth - Optimized for Colab
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-deps "xformers<0.0.27" "trl<0.9.0" peft accelerate bitsandbytes triton

# Additional dependencies
!pip install datasets wandb huggingface_hub

In [ ]:
# Verify GPU
import torch
print(f"🔥 PyTorch version: {torch.__version__}")
print(f"🎮 CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"🖥️ GPU: {torch.cuda.get_device_name(0)}")
    print(f"💾 VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    print("❌ No GPU detected! Go to Runtime > Change runtime type > GPU")

## 🔐 Step 2: Setup Authentication

Login to Hugging Face to download Llama 3 and save your model.

In [ ]:
from huggingface_hub import login
from google.colab import userdata

# Option 1: Use Colab secrets (recommended)
try:
    hf_token = userdata.get('HF_TOKEN')
    login(token=hf_token)
    print("✅ Logged in via Colab secrets")
except:
    # Option 2: Manual login
    print("💡 Add HF_TOKEN to Colab secrets, or login manually:")
    login()

## 🦙 Step 3: Load Llama 3.1 8B with Unsloth

Unsloth automatically applies optimizations for 2x faster training.

In [ ]:
from unsloth import FastLanguageModel
import torch

# Configuration
max_seq_length = 2048  # Can go up to 8192 for Llama 3, but uses more VRAM
dtype = None  # Auto-detect (float16 for T4, bfloat16 for A100)
load_in_4bit = True  # Use 4-bit quantization to save memory

# Model options (uncomment one):
# model_name = "unsloth/llama-3-8b-bnb-4bit"           # Llama 3 8B
model_name = "unsloth/Meta-Llama-3.1-8B-bnb-4bit"     # Llama 3.1 8B (recommended)
# model_name = "unsloth/mistral-7b-v0.3-bnb-4bit"     # Mistral 7B (alternative)
# model_name = "unsloth/Phi-3-mini-4k-instruct-bnb-4bit"  # Phi-3 (smaller, faster)

print(f"📥 Loading {model_name}...")
print("   This may take 2-5 minutes on first run.")

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=model_name,
    max_seq_length=max_seq_length,
    dtype=dtype,
    load_in_4bit=load_in_4bit,
)

print(f"✅ Model loaded successfully!")
print(f"   Memory used: {torch.cuda.memory_allocated()/1e9:.2f} GB")

## 🔧 Step 4: Configure LoRA Adapters

LoRA (Low-Rank Adaptation) allows efficient fine-tuning by only training small adapter layers.

In [ ]:
# Add LoRA adapters
model = FastLanguageModel.get_peft_model(
    model,
    r=16,  # LoRA rank (higher = more capacity, more memory). Try 8, 16, 32, 64
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",  # Attention layers
        "gate_proj", "up_proj", "down_proj",      # MLP layers
    ],
    lora_alpha=16,  # LoRA scaling factor
    lora_dropout=0,  # 0 is optimized by Unsloth
    bias="none",     # "none" is optimized
    use_gradient_checkpointing="unsloth",  # Saves 30% memory
    random_state=42,
    use_rslora=False,  # Rank-stabilized LoRA (experimental)
    loftq_config=None,
)

# Print trainable parameters
def print_trainable_parameters(model):
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    total = sum(p.numel() for p in model.parameters())
    print(f"🎯 Trainable parameters: {trainable:,} ({100 * trainable / total:.2f}%)")
    print(f"📊 Total parameters: {total:,}")

print_trainable_parameters(model)

## 📂 Step 5: Upload and Load Your Dataset

Upload your `finetuning_data.jsonl` or `reasoning_entries.json` file.

In [ ]:
from google.colab import files
import json
import os

print("📤 Please upload your dataset file:")
print("   - finetuning_data.jsonl (recommended)")
print("   - OR reasoning_entries.json")
print()

uploaded = files.upload()

# Get the uploaded filename
uploaded_filename = list(uploaded.keys())[0]
print(f"\n✅ Uploaded: {uploaded_filename}")

In [ ]:
from datasets import Dataset, load_dataset
import json

# Define the chat template for Llama 3
LLAMA3_CHAT_TEMPLATE = """{% for message in messages %}
{% if message['role'] == 'system' %}
<|begin_of_text|><|start_header_id|>system<|end_header_id|>

{{ message['content'] }}<|eot_id|>
{% elif message['role'] == 'user' %}
<|start_header_id|>user<|end_header_id|>

{{ message['content'] }}<|eot_id|>
{% elif message['role'] == 'assistant' %}
<|start_header_id|>assistant<|end_header_id|>

{{ message['content'] }}<|eot_id|>
{% endif %}
{% endfor %}
{% if add_generation_prompt %}
<|start_header_id|>assistant<|end_header_id|>

{% endif %}"""

# Set the chat template
tokenizer.chat_template = LLAMA3_CHAT_TEMPLATE

def load_legal_dataset(filename):
    """Load dataset from either JSONL or JSON format."""
    
    if filename.endswith('.jsonl'):
        # Load JSONL (fine-tuning format)
        data = []
        with open(filename, 'r', encoding='utf-8') as f:
            for line in f:
                data.append(json.loads(line))
        print(f"📊 Loaded {len(data)} entries from JSONL")
        return Dataset.from_list(data)
    
    elif filename.endswith('.json'):
        # Load JSON (reasoning entries format)
        with open(filename, 'r', encoding='utf-8') as f:
            raw_data = json.load(f)
        
        # Convert to chat format
        formatted_data = []
        entries = raw_data if isinstance(raw_data, list) else raw_data.get('reasoning_entries', [])
        
        for entry in entries:
            # Build the assistant response with reasoning
            response = build_reasoning_response(entry)
            
            formatted_data.append({
                "messages": [
                    {
                        "role": "system",
                        "content": "You are a Sri Lankan property law expert. Provide step-by-step legal reasoning with references to relevant statutes and case law. Always explain your reasoning clearly."
                    },
                    {
                        "role": "user",
                        "content": entry.get("question", "")
                    },
                    {
                        "role": "assistant",
                        "content": response
                    }
                ]
            })
        
        print(f"📊 Converted {len(formatted_data)} entries from JSON")
        return Dataset.from_list(formatted_data)
    
    else:
        raise ValueError(f"Unsupported file format: {filename}")


def build_reasoning_response(entry):
    """Build a structured reasoning response from entry data."""
    
    response_parts = []
    
    # Short answer
    if entry.get("short_answer"):
        response_parts.append(f"**Answer:** {entry['short_answer']}")
        response_parts.append("")
    
    # Reasoning chain
    if entry.get("reasoning_chain"):
        response_parts.append("**Step-by-Step Legal Analysis:**")
        response_parts.append("")
        for step in entry["reasoning_chain"]:
            step_num = step.get("step_number", "")
            action = step.get("action", "")
            legal_basis = step.get("legal_basis", "")
            result = step.get("result", "")
            
            response_parts.append(f"**Step {step_num}: {action}**")
            if legal_basis:
                response_parts.append(f"- Legal Basis: {legal_basis}")
            if result:
                response_parts.append(f"- Finding: {result}")
            response_parts.append("")
    
    # IRAC Analysis
    irac = entry.get("irac_analysis", {})
    if irac:
        response_parts.append("**IRAC Analysis:**")
        response_parts.append("")
        if irac.get("issue"):
            response_parts.append(f"- **Issue:** {irac['issue']}")
        if irac.get("rule"):
            response_parts.append(f"- **Rule:** {irac['rule']}")
        if irac.get("application"):
            response_parts.append(f"- **Application:** {irac['application']}")
        if irac.get("conclusion"):
            response_parts.append(f"- **Conclusion:** {irac['conclusion']}")
        response_parts.append("")
    
    # Legal References
    refs = entry.get("legal_references", [])
    if refs:
        response_parts.append("**Legal References:**")
        for ref in refs:
            statute = ref.get("statute", "")
            section = ref.get("section", "")
            relevance = ref.get("relevance", "")
            response_parts.append(f"- {statute}, {section}: {relevance}")
        response_parts.append("")
    
    # Example scenario
    scenario = entry.get("example_scenario", {})
    if scenario:
        response_parts.append("**Practical Example:**")
        if scenario.get("facts"):
            response_parts.append(f"- Facts: {scenario['facts']}")
        if scenario.get("analysis"):
            response_parts.append(f"- Analysis: {scenario['analysis']}")
        if scenario.get("outcome"):
            response_parts.append(f"- Outcome: {scenario['outcome']}")
    
    return "\n".join(response_parts)


# Load the dataset
dataset = load_legal_dataset(uploaded_filename)
print(f"\n✅ Dataset ready with {len(dataset)} examples")
print(f"\n📝 Sample entry:")
print(dataset[0])

In [ ]:
# Format dataset for training
def formatting_prompts_func(examples):
    """Format examples using the chat template."""
    convos = examples["messages"]
    texts = []
    for convo in convos:
        text = tokenizer.apply_chat_template(
            convo, 
            tokenize=False, 
            add_generation_prompt=False
        )
        texts.append(text)
    return {"text": texts}

# Apply formatting
dataset = dataset.map(formatting_prompts_func, batched=True)

print("✅ Dataset formatted for training")
print(f"\n📝 Formatted sample (first 500 chars):")
print(dataset[0]["text"][:500] + "...")

## 🎯 Step 6: Configure Training Parameters

These settings are optimized for Colab T4 GPU with 15GB VRAM.

In [ ]:
from trl import SFTTrainer
from transformers import TrainingArguments
from unsloth import is_bfloat16_supported

# Training configuration
training_args = TrainingArguments(
    # Output
    output_dir="./legalvision_model",
    
    # Training hyperparameters
    num_train_epochs=3,                    # Number of epochs (3-5 recommended)
    per_device_train_batch_size=2,         # Batch size (reduce if OOM)
    gradient_accumulation_steps=4,         # Effective batch = 2 * 4 = 8
    
    # Learning rate
    learning_rate=2e-4,                    # LoRA learning rate (2e-4 to 5e-4)
    lr_scheduler_type="linear",            # linear, cosine, constant
    warmup_steps=10,                       # Warmup steps
    
    # Optimization
    optim="adamw_8bit",                    # 8-bit Adam (saves memory)
    weight_decay=0.01,                     # Regularization
    max_grad_norm=0.3,                     # Gradient clipping
    
    # Precision
    fp16=not is_bfloat16_supported(),      # Use fp16 on T4
    bf16=is_bfloat16_supported(),          # Use bf16 on A100
    
    # Logging
    logging_steps=10,                      # Log every N steps
    logging_dir="./logs",
    
    # Saving
    save_strategy="epoch",                 # Save after each epoch
    save_total_limit=2,                    # Keep only last 2 checkpoints
    
    # Misc
    seed=42,
    report_to="none",                      # Disable W&B (set to "wandb" to enable)
)

print("✅ Training configuration set")
print(f"   - Epochs: {training_args.num_train_epochs}")
print(f"   - Batch size: {training_args.per_device_train_batch_size}")
print(f"   - Gradient accumulation: {training_args.gradient_accumulation_steps}")
print(f"   - Effective batch size: {training_args.per_device_train_batch_size * training_args.gradient_accumulation_steps}")
print(f"   - Learning rate: {training_args.learning_rate}")

In [ ]:
# Create trainer
trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=dataset,
    dataset_text_field="text",
    max_seq_length=max_seq_length,
    dataset_num_proc=2,
    packing=False,  # Can set to True for shorter sequences
    args=training_args,
)

print("✅ Trainer initialized")

## 🚀 Step 7: Start Training!

Training time depends on dataset size:
- 50 examples: ~10-15 minutes
- 200 examples: ~30-45 minutes
- 500 examples: ~1-2 hours

In [ ]:
# Check memory before training
gpu_stats = torch.cuda.get_device_properties(0)
start_gpu_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
max_memory = round(gpu_stats.total_memory / 1024 / 1024 / 1024, 3)
print(f"🖥️ GPU: {gpu_stats.name}")
print(f"💾 Memory reserved: {start_gpu_memory} GB / {max_memory} GB")
print(f"\n🚀 Starting training...\n")

# Train!
trainer_stats = trainer.train()

# Print final stats
print(f"\n" + "="*50)
print("✅ TRAINING COMPLETE!")
print("="*50)
print(f"⏱️ Training time: {trainer_stats.metrics['train_runtime']:.2f} seconds")
print(f"📊 Training loss: {trainer_stats.metrics['train_loss']:.4f}")
print(f"🔢 Total steps: {trainer_stats.metrics['train_steps']}")

# Memory stats
used_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
print(f"💾 Peak memory: {used_memory} GB / {max_memory} GB")

## 🧪 Step 8: Test the Fine-tuned Model

Let's test the model with some legal questions!

In [ ]:
# Enable inference mode
FastLanguageModel.for_inference(model)

def ask_legal_question(question, max_new_tokens=1024):
    """Ask a legal question and get a reasoned response."""
    
    messages = [
        {
            "role": "system",
            "content": "You are a Sri Lankan property law expert. Provide step-by-step legal reasoning with references to relevant statutes and case law. Always explain your reasoning clearly."
        },
        {
            "role": "user",
            "content": question
        }
    ]
    
    # Format input
    inputs = tokenizer.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=True,
        return_tensors="pt"
    ).to("cuda")
    
    # Generate
    outputs = model.generate(
        input_ids=inputs,
        max_new_tokens=max_new_tokens,
        use_cache=True,
        temperature=0.7,
        top_p=0.9,
        do_sample=True,
    )
    
    # Decode
    response = tokenizer.decode(outputs[0], skip_special_tokens=True)
    
    # Extract assistant response
    if "assistant" in response:
        response = response.split("assistant")[-1].strip()
    
    return response

print("✅ Inference mode enabled")
print("\n" + "="*60)

In [ ]:
# Test Question 1: Property Transfer
question1 = "What are the legal requirements for a valid property transfer in Sri Lanka?"

print("🔍 QUESTION 1:")
print(question1)
print("\n" + "-"*60)
print("📜 RESPONSE:")
print(ask_legal_question(question1))
print("\n" + "="*60)

In [ ]:
# Test Question 2: Prescription
question2 = "How can someone acquire land through prescription in Sri Lanka?"

print("🔍 QUESTION 2:")
print(question2)
print("\n" + "-"*60)
print("📜 RESPONSE:")
print(ask_legal_question(question2))
print("\n" + "="*60)

In [ ]:
# Test Question 3: Partition
question3 = "Can a minority shareholder force partition of co-owned property?"

print("🔍 QUESTION 3:")
print(question3)
print("\n" + "-"*60)
print("📜 RESPONSE:")
print(ask_legal_question(question3))
print("\n" + "="*60)

In [ ]:
# Interactive testing
print("💬 Interactive Legal Q&A")
print("   Type your question and press Enter")
print("   Type 'quit' to exit\n")

while True:
    question = input("\n📝 Your question: ")
    if question.lower() in ['quit', 'exit', 'q']:
        print("\n👋 Goodbye!")
        break
    
    print("\n🤔 Thinking...")
    response = ask_legal_question(question)
    print(f"\n📜 Response:\n{response}")
    print("\n" + "-"*60)

## 💾 Step 9: Save the Model

Multiple saving options available:

In [ ]:
# Option 1: Save LoRA adapters only (smallest, recommended)
print("💾 Saving LoRA adapters...")
model.save_pretrained("legalvision_lora")
tokenizer.save_pretrained("legalvision_lora")
print("✅ Saved to ./legalvision_lora/")

In [ ]:
# Option 2: Save merged 16-bit model (for vLLM, TensorRT)
print("💾 Saving merged 16-bit model...")
model.save_pretrained_merged(
    "legalvision_merged_16bit",
    tokenizer,
    save_method="merged_16bit",
)
print("✅ Saved to ./legalvision_merged_16bit/")

In [ ]:
# Option 3: Save as GGUF for llama.cpp (local inference)
print("💾 Saving as GGUF (Q4_K_M quantization)...")
model.save_pretrained_gguf(
    "legalvision_gguf",
    tokenizer,
    quantization_method="q4_k_m"  # Options: q4_k_m, q5_k_m, q8_0, f16
)
print("✅ Saved to ./legalvision_gguf/")

In [ ]:
# Option 4: Push to Hugging Face Hub
HF_USERNAME = "your-username"  # Change this!
MODEL_NAME = "legalvision-llama3-srilanka-law"

print(f"📤 Pushing to Hugging Face Hub as {HF_USERNAME}/{MODEL_NAME}...")

# Push LoRA adapters
model.push_to_hub(
    f"{HF_USERNAME}/{MODEL_NAME}-lora",
    token=hf_token,
    private=True,  # Set to False to make public
)
tokenizer.push_to_hub(
    f"{HF_USERNAME}/{MODEL_NAME}-lora",
    token=hf_token,
    private=True,
)

print(f"✅ Pushed to https://huggingface.co/{HF_USERNAME}/{MODEL_NAME}-lora")

## 📥 Step 10: Download Model Files

Download the trained model to your local machine.

In [ ]:
# Zip and download LoRA adapters
!zip -r legalvision_lora.zip legalvision_lora/

from google.colab import files
files.download('legalvision_lora.zip')

print("\n✅ Download started! Check your browser downloads.")

In [ ]:
# Download GGUF file (if created)
import os
if os.path.exists("legalvision_gguf"):
    # Find the GGUF file
    gguf_files = [f for f in os.listdir("legalvision_gguf") if f.endswith('.gguf')]
    if gguf_files:
        gguf_path = f"legalvision_gguf/{gguf_files[0]}"
        print(f"📥 Downloading {gguf_path}...")
        files.download(gguf_path)
    else:
        print("⚠️ No GGUF file found")
else:
    print("⚠️ GGUF folder not found. Run the GGUF save cell first.")

## 🔄 Step 11: Load and Use the Model Later

Code to load your fine-tuned model in a new session.

In [ ]:
# Code to load the model later (copy this for future use)

LOAD_CODE = '''
# Load fine-tuned LegalVision model
from unsloth import FastLanguageModel

# Option 1: Load from local LoRA adapters
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="./legalvision_lora",  # or path to uploaded folder
    max_seq_length=2048,
    dtype=None,
    load_in_4bit=True,
)
FastLanguageModel.for_inference(model)

# Option 2: Load from Hugging Face Hub
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="your-username/legalvision-llama3-srilanka-law-lora",
    max_seq_length=2048,
    dtype=None,
    load_in_4bit=True,
)
FastLanguageModel.for_inference(model)

# Use the model
messages = [
    {"role": "system", "content": "You are a Sri Lankan property law expert."},
    {"role": "user", "content": "What is prescription in property law?"}
]

inputs = tokenizer.apply_chat_template(messages, tokenize=True, add_generation_prompt=True, return_tensors="pt").to("cuda")
outputs = model.generate(input_ids=inputs, max_new_tokens=512, temperature=0.7)
print(tokenizer.decode(outputs[0], skip_special_tokens=True))
'''

print("📋 Code to load your model in future sessions:")
print("="*60)
print(LOAD_CODE)

---

## ✅ Summary

You have successfully:

1. ✅ Loaded Llama 3.1 8B with 4-bit quantization
2. ✅ Added LoRA adapters for efficient fine-tuning
3. ✅ Loaded your legal reasoning dataset
4. ✅ Fine-tuned the model on Sri Lankan property law
5. ✅ Tested the model with legal questions
6. ✅ Saved the model in multiple formats

### 📊 Model Outputs:
- `legalvision_lora/` - LoRA adapters (~50-100MB)
- `legalvision_merged_16bit/` - Full model in 16-bit (~16GB)
- `legalvision_gguf/` - GGUF for llama.cpp (~4-8GB)

### 🚀 Next Steps:
1. Integrate with your GraphRAG system
2. Build a FastAPI endpoint for inference
3. Create a React frontend
4. Deploy to cloud (AWS, GCP, etc.)

---

**LegalVision Project - S. Sivanuja**  
Sri Lanka Institute of Information Technology